# 聚合 baseline comparison score shards

该 Notebook 只负责聚合某一个 baseline 的 shard score records, 不执行 baseline detector。

In [ ]:
# 可修改配置区
REPO_URL = "https://github.com/RICHAAARC/TSTW-VW.git"  # Colab 冷启动使用的仓库 clone URL
REPO_DIR = "/content/TSTW-VW"  # Colab 本地仓库根目录
GIT_REF = "explicit-sync"  # 使用的项目分支
DRIVE_MOUNT = "/content/drive"  # Google Drive 挂载点
DRIVE_RESULT_ROOT = "/content/drive/MyDrive/TSTW/results"  # Drive results 根目录
BASELINE_NAME = "external_hidden_framewise"  # 要聚合的 baseline 名称, 例如 external_videoseal/external_rivagan/external_hidden_framewise
BASELINE_SCORE_RECORD_PATHS = []  # 可选: 当前 baseline 所有 shard 的 baseline_formal_score_records.jsonl 路径。留空时自动从 <baseline>/shard_runs 搜索。
AUTO_RESOLVE_BASELINE_SCORE_RECORD_PATHS = True  # 是否按 BASELINE_NAME 自动搜索 Drive 中的 shard_runs。
REQUIRED_SHARD_SHORT_COMMIT = ""  # 可选: 只聚合同一 short commit 的 shard, 留空表示自动选择最新 commit 分组。
REQUIRED_SHARD_COUNT = None  # 可选: 期望 shard 总数, None 表示由自动搜索到的完整分组决定。
BASELINE_SCORE_AGGREGATION_RUN_ROOT = "/content/TSTW_runtime/runs/baseline_score_records_aggregation"  # 聚合本地 run_root
TARGET_FPR = 0.01  # 论文级 1% FPR 时使用 0.01
OVERWRITE_DRIVE_RESULT = True  # Drive 聚合目录已存在时是否覆盖


In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys


def run_command(command, cwd=None):
    print("$", " ".join(map(str, command)), flush=True)
    completed = subprocess.run(command, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr, flush=True)
    if completed.returncode != 0:
        raise RuntimeError("命令失败: " + " ".join(map(str, command)))
    return completed



def parse_json_process_stdout(stdout_text):
    """解析仓库脚本输出的 JSON。

    脚本通常使用 indent=2 打印多行 JSON, 因此不能只读取最后一行。
    该函数从完整 stdout 中定位第一个 JSON 对象并解析, 便于 Colab cell 稳定接收结果。
    """
    resolved_text = str(stdout_text).strip()
    if not resolved_text:
        raise ValueError("脚本 stdout 为空, 无法解析 JSON 结果。")
    json_start = resolved_text.find("{")
    if json_start < 0:
        raise ValueError("脚本 stdout 中未找到 JSON 对象。")
    return json.loads(resolved_text[json_start:])

def split_paths(raw_paths):
    if isinstance(raw_paths, (list, tuple)):
        return [str(item).strip() for item in raw_paths if str(item).strip()]
    return [item.strip() for item in re.split(r"[;,]", str(raw_paths)) if item.strip()]


def parse_baseline_shard_run_name(path):
    """解析 baseline shard run 目录名中的 shard_count、shard_index 和 short_commit。"""
    match = re.search(r"_sc(?P<shard_count>\d+)_si(?P<shard_index>\d+)_(?P<short_commit>[0-9a-fA-F]+|unknown)$", Path(path).name)
    if not match:
        return None
    return {
        "shard_count": int(match.group("shard_count")),
        "shard_index": int(match.group("shard_index")),
        "short_commit": match.group("short_commit"),
    }


def discover_baseline_score_record_groups():
    """按 BASELINE_NAME 在 <baseline>/shard_runs 下搜索可聚合的 score records 分组。"""
    shard_runs_root = Path(DRIVE_RESULT_ROOT) / "baseline_comparison_gate" / BASELINE_NAME / "shard_runs"
    if not shard_runs_root.exists():
        raise FileNotFoundError(f"未找到 baseline shard_runs 目录: {shard_runs_root}")
    groups = {}
    for run_root in sorted(shard_runs_root.glob("baseline_comparison_formal_scoring_execution_*")):
        if not run_root.is_dir():
            continue
        parsed = parse_baseline_shard_run_name(run_root)
        if parsed is None:
            continue
        if REQUIRED_SHARD_SHORT_COMMIT and parsed["short_commit"] != REQUIRED_SHARD_SHORT_COMMIT:
            continue
        if REQUIRED_SHARD_COUNT is not None and parsed["shard_count"] != int(REQUIRED_SHARD_COUNT):
            continue
        record_path = run_root / "records" / "baseline_formal_score_records.jsonl"
        manifest_path = run_root / "artifacts" / "baseline_comparison_formal_scoring_execution_manifest.json"
        if not record_path.exists() or not manifest_path.exists():
            continue
        group_key = (parsed["short_commit"], parsed["shard_count"])
        groups.setdefault(group_key, {})[parsed["shard_index"]] = {
            "run_root": run_root,
            "record_path": record_path,
            "manifest_path": manifest_path,
            **parsed,
        }
    complete_groups = []
    for (short_commit, shard_count), items_by_index in groups.items():
        expected_indexes = set(range(shard_count))
        actual_indexes = set(items_by_index)
        complete_groups.append({
            "short_commit": short_commit,
            "shard_count": shard_count,
            "shard_indexes": sorted(actual_indexes),
            "complete": actual_indexes == expected_indexes,
            "missing_shard_indexes": sorted(expected_indexes - actual_indexes),
            "record_paths": [items_by_index[index]["record_path"] for index in sorted(actual_indexes)],
            "latest_mtime": max(item["run_root"].stat().st_mtime for item in items_by_index.values()),
        })
    complete_groups.sort(key=lambda item: (item["complete"], item["latest_mtime"]), reverse=True)
    return complete_groups


def resolve_baseline_score_record_paths():
    """解析聚合输入路径。手动路径优先, 否则按 baseline 自动搜索完整 shard 分组。"""
    manual_paths = split_paths(BASELINE_SCORE_RECORD_PATHS)
    if manual_paths:
        return manual_paths, {"mode": "manual", "record_paths": manual_paths}
    if not AUTO_RESOLVE_BASELINE_SCORE_RECORD_PATHS:
        raise ValueError("BASELINE_SCORE_RECORD_PATHS 为空且 AUTO_RESOLVE_BASELINE_SCORE_RECORD_PATHS=False。")
    groups = discover_baseline_score_record_groups()
    if not groups:
        raise FileNotFoundError(f"未找到 {BASELINE_NAME} 的 shard score records。")
    selected = next((group for group in groups if group["complete"]), groups[0])
    if not selected["complete"]:
        raise RuntimeError({
            "reason": "baseline_shard_group_incomplete",
            "baseline": BASELINE_NAME,
            "selected_group": {
                **selected,
                "record_paths": [str(path) for path in selected["record_paths"]],
            },
        })
    return [str(path) for path in selected["record_paths"]], {
        "mode": "auto",
        "baseline": BASELINE_NAME,
        "selected_group": {
            **selected,
            "record_paths": [str(path) for path in selected["record_paths"]],
        },
        "candidate_group_count": len(groups),
    }


## 1. 挂载 Drive 并获取仓库

In [ ]:
try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
except Exception as exc:
    print(f"非 Colab 环境或 Drive 挂载失败: {exc}")
repo_path = Path(REPO_DIR)
if repo_path.exists():
    run_command(["git", "fetch", "--depth", "1", "origin", GIT_REF], cwd=repo_path)
    run_command(["git", "checkout", GIT_REF], cwd=repo_path)
    run_command(["git", "pull", "--ff-only", "origin", GIT_REF], cwd=repo_path)
else:
    run_command(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, REPO_DIR])
short_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True).strip()
print({"short_commit": short_commit})


## 2. 聚合 score records

In [ ]:
record_paths, record_path_resolution = resolve_baseline_score_record_paths()
print(json.dumps(record_path_resolution, ensure_ascii=False, indent=2, default=str))
aggregation_args = [
    sys.executable,
    "scripts/prepare_baselines/aggregate_baseline_score_records.py",
    "--run-root", BASELINE_SCORE_AGGREGATION_RUN_ROOT,
    "--result-root", DRIVE_RESULT_ROOT,
    "--baseline-name", BASELINE_NAME,
    "--target-fpr", str(TARGET_FPR),
    "--short-commit", short_commit,
]
if OVERWRITE_DRIVE_RESULT:
    aggregation_args.append("--overwrite")
for record_path in record_paths:
    aggregation_args.extend(["--record-path", record_path])
completed = run_command(aggregation_args, cwd=REPO_DIR)
aggregation_payload = parse_json_process_stdout(completed.stdout)
print(aggregation_payload)


## 3. 最终完成性判断

In [ ]:
materialized_path = Path(aggregation_payload["materialized_path"])
required_paths = {
    "aggregation_manifest": materialized_path / "artifacts" / "baseline_score_records_aggregation_manifest.json",
    "baseline_comparison_table": materialized_path / "tables" / "baseline_comparison_table.csv",
    "baseline_attack_breakdown": materialized_path / "tables" / "baseline_attack_breakdown.csv",
    "baseline_threshold_table": materialized_path / "thresholds" / "baseline_threshold_table.csv",
    "baseline_runtime_table": materialized_path / "tables" / "baseline_runtime_table.csv",
}
completion_summary = {
    "result_flow": "baseline_shard_aggregate",
    "baseline": BASELINE_NAME,
    "target_fpr": TARGET_FPR,
    "drive_aggregated_root": str(materialized_path),
    "input_record_paths": record_paths,
    "record_path_resolution": record_path_resolution,
    "required_paths": {name: path.exists() for name, path in required_paths.items()},
    "ready_for_paper_artifact_gate": True,
}
completion_summary["status"] = all(completion_summary["required_paths"].values())
print(json.dumps(completion_summary, ensure_ascii=False, indent=2))
if not completion_summary["status"]:
    raise RuntimeError(completion_summary)
